# Exploratory Data Analysis


In [70]:
import numpy as np
import pandas as pd
import yfinance as yf
from market_ml.data_loader import DataLoader
import matplotlib.pyplot as plt
import plotly.express as px
from market_ml.config import * 

# Data Loading 


In [71]:
dl = DataLoader(tickers=TICKERS, period="10y", interval="1d")
market_data = dl.load_data()
sectors = market_data['Sector'].unique()

sectors

Data loaded for ['AAPL', 'MSFT', 'NVDA', 'META', 'GOOGL', 'JPM', 'GS', 'BAC', 'MS', 'BLK', 'JNJ', 'UNH', 'ABBV', 'PFE', 'MRK', 'XOM', 'CVX', 'COP', 'SLB', 'EOG', 'AMZN', 'COST', 'WMT', 'HD', 'MCD']


<ArrowStringArray>
['Technology', 'Healthcare', 'Consumer', 'Financials', 'Energy']
Length: 5, dtype: str

In [72]:
market_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 62825 entries, 0 to 62824
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    62825 non-null  datetime64[ms]
 1   Ticker  62825 non-null  str           
 2   Sector  62825 non-null  str           
 3   Close   62825 non-null  float64       
 4   High    62825 non-null  float64       
 5   Low     62825 non-null  float64       
 6   Open    62825 non-null  float64       
 7   Volume  62825 non-null  int64         
dtypes: datetime64[ms](1), float64(4), int64(1), str(2)
memory usage: 4.6 MB


In [73]:
print(market_data.shape)
print(market_data['Ticker'].unique())
print(market_data['Sector'].value_counts())



(62825, 8)
<ArrowStringArray>
[ 'AAPL',  'ABBV',  'AMZN',   'BAC',   'BLK',   'COP',  'COST',   'CVX',
   'EOG', 'GOOGL',    'GS',    'HD',   'JNJ',   'JPM',   'MCD',  'META',
   'MRK',    'MS',  'MSFT',  'NVDA',   'PFE',   'SLB',   'UNH',   'WMT',
   'XOM']
Length: 25, dtype: str
Sector
Technology    12565
Healthcare    12565
Consumer      12565
Financials    12565
Energy        12565
Name: count, dtype: int64


In [74]:
market_data.groupby("Ticker").agg(
    Rows=("Ticker", "size"),
    Start=("Date", "min"),
    End=("Date", "max")
)

,Rows,Start,End
Ticker,,,
AAPL,2513,2016-09-28,2026-09-28
ABBV,2513,2016-09-28,2026-09-28
AMZN,2513,2016-09-28,2026-09-28
BAC,2513,2016-09-28,2026-09-28
BLK,2513,2016-09-28,2026-09-28
COP,2513,2016-09-28,2026-09-28
COST,2513,2016-09-28,2026-09-28
CVX,2513,2016-09-28,2026-09-28
EOG,2513,2016-09-28,2026-09-28


Great, no obvious inbalance in the data 

# Sector Daily Returns

In [75]:
market_data["Daily_Return"] = market_data.groupby("Ticker")["Close"].pct_change()

market_data["Daily_Return"].describe()

count    62800.000000
mean         0.000819
std          0.019273
min         -0.320072
25%         -0.007817
50%          0.000768
75%          0.009565
max          0.298067
Name: Daily_Return, dtype: float64

In [83]:
for sector in sectors:
    sector_data = market_data[market_data['Sector']==sector]
    fig = px.histogram(sector_data, x="Daily_Return", nbins=100, facet_col='Ticker',title=f'{sector} Daily Return Distributions' )
    fig.show()
    

#### Daily returns mostly centered around 0.0 but some large values in the tails, showing overall the daily returns are small with some rarer big changes. 

# Sector Volatility

In [86]:
market_data["Volatility_20D"] = market_data.groupby("Ticker")["Daily_Return"].rolling(window=20).std().reset_index(level=0, drop=True)

In [91]:
for sector in sectors:
    sector_data = market_data[market_data['Sector']==sector]
    fig = px.line(sector_data, x="Date", y="Volatility_20D", color="Ticker", title=f' {sector} 20 Day Rolling Volatility')
    fig.update_layout(yaxis_title="Volatility", xaxis_title="Date")
    fig.show()
    

#### Rolling 20 day volatility varies with time, peaking at early 2020 (pandemic) and early 2025 (Tarrifs). Shows a need for volality to be included in the modelling. 

# Drawdown

In [92]:
market_data["Cumulative_Returns"] = market_data.groupby("Ticker")["Daily_Return"].transform(lambda x: (1+x).cumprod())

market_data["Highest_Peak"] = market_data.groupby("Ticker")["Cumulative_Returns"].cummax()

market_data["Drawdown"] = (market_data["Cumulative_Returns"]-market_data["Highest_Peak"])/market_data["Highest_Peak"]

In [94]:
for sector in sectors:
    sector_data = market_data[market_data['Sector']==sector]
    fig = px.line(sector_data, x="Date", y="Drawdown", color="Ticker", title=f"{sector} Drawdown by Stock")
    fig.show()

In [97]:
#max drawdown
for sector in sectors:
    sector_data = market_data[market_data['Sector']==sector]

    
    max_drawdown = sector_data.groupby('Ticker')['Drawdown'].min().sort_values()
    print('--'*10)
    print(sector)
    display(max_drawdown)
    

--------------------
Technology


Ticker
META    -0.767361
NVDA    -0.663351
GOOGL   -0.443201
AAPL    -0.385159
MSFT    -0.371485
Name: Drawdown, dtype: float64

--------------------
Healthcare


Ticker
UNH    -0.613909
PFE    -0.589620
ABBV   -0.450898
MRK    -0.434434
JNJ    -0.273663
Name: Drawdown, dtype: float64

--------------------
Consumer


Ticker
AMZN   -0.561453
HD     -0.379937
MCD    -0.368986
COST   -0.314024
WMT    -0.257397
Name: Drawdown, dtype: float64

--------------------
Financials


Ticker
MS    -0.513258
BAC   -0.489468
GS    -0.487488
BLK   -0.439017
JPM   -0.436265
Name: Drawdown, dtype: float64

--------------------
Energy


Ticker
SLB   -0.842851
EOG   -0.771317
COP   -0.706610
XOM   -0.610066
CVX   -0.557739
Name: Drawdown, dtype: float64

# Correlation

In [45]:
market_returns = market_data.pivot(index="Date", columns='Ticker', values='Daily_Return')

market_correlation = market_returns.corr()

market_correlation

Ticker,AAPL,META,MSFT,NVDA
Ticker,,,,
AAPL,1.000000,0.502568,0.634666,0.523204
META,0.502568,1.000000,0.552808,0.488287
MSFT,0.634666,0.552808,1.000000,0.591230
NVDA,0.523204,0.488287,0.591230,1.000000


In [99]:
for sector in sectors:
    sector_data = market_data[market_data['Sector']==sector]
    sector_returns = sector_data.pivot(index="Date", columns='Ticker', values='Daily_Return')

    sector_correlation = sector_returns.corr()
    print(sector)
    display(sector_correlation)


Technology


Ticker,AAPL,GOOGL,META,MSFT,NVDA
Ticker,,,,,
AAPL,1.000000,0.579321,0.502060,0.634914,0.523223
GOOGL,0.579321,1.000000,0.587974,0.648046,0.524880
META,0.502060,0.587974,1.000000,0.552759,0.487405
MSFT,0.634914,0.648046,0.552759,1.000000,0.590727
NVDA,0.523223,0.524880,0.487405,0.590727,1.000000


Healthcare


Ticker,ABBV,JNJ,MRK,PFE,UNH
Ticker,,,,,
ABBV,1.000000,0.449179,0.442793,0.448810,0.322861
JNJ,0.449179,1.000000,0.502923,0.497862,0.349628
MRK,0.442793,0.502923,1.000000,0.472569,0.307908
PFE,0.448810,0.497862,0.472569,1.000000,0.296207
UNH,0.322861,0.349628,0.307908,0.296207,1.000000


Consumer


Ticker,AMZN,COST,HD,MCD,WMT
Ticker,,,,,
AMZN,1.000000,0.397720,0.395969,0.219198,0.239518
COST,0.397720,1.000000,0.487324,0.352995,0.578999
HD,0.395969,0.487324,1.000000,0.489503,0.381766
MCD,0.219198,0.352995,0.489503,1.000000,0.297639
WMT,0.239518,0.578999,0.381766,0.297639,1.000000


Financials


Ticker,BAC,BLK,GS,JPM,MS
Ticker,,,,,
BAC,1.000000,0.658401,0.795369,0.880439,0.820171
BLK,0.658401,1.000000,0.666169,0.654369,0.695936
GS,0.795369,0.666169,1.000000,0.804048,0.864370
JPM,0.880439,0.654369,0.804048,1.000000,0.808276
MS,0.820171,0.695936,0.864370,0.808276,1.000000


Energy


Ticker,COP,CVX,EOG,SLB,XOM
Ticker,,,,,
COP,1.000000,0.821891,0.852370,0.753606,0.814072
CVX,0.821891,1.000000,0.743947,0.731113,0.835992
EOG,0.852370,0.743947,1.000000,0.741980,0.780452
SLB,0.753606,0.731113,0.741980,1.000000,0.739491
XOM,0.814072,0.835992,0.780452,0.739491,1.000000
